# scipy.optimize で学ぶ最適化 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、SciPy の最適化モジュール
**scipy.optimize** を使って、経済学・経営学で登場する最適化問題（利潤最大化、効用最大化、生産計画、輸送問題、
ポートフォリオ選択など）を **数値的に** 解く方法を学ぶチュートリアルです。

## 対象者
- Python と NumPy の基本を理解している方
- 経済数学（微分・制約付き最適化）を学んだ、または SymPy のチュートリアルを終えた方
- 「式では解けない」問題をコンピュータで解きたい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. 最適化問題とは：目的関数・変数・制約
2. 1 変数の最適化：`minimize_scalar`
3. 多変数の最適化：`minimize`
4. 制約付き最適化：SLSQP と予算制約
5. 線形計画法：`linprog`（生産計画・輸送問題・食事問題）
6. 感度分析：シャドウプライス（双対値）
7. 整数計画法：`milp`（プロジェクト選択）
8. cvxpy でポートフォリオ最適化
9. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 各章の最後に **練習問題** があります。「解答欄」に自分でコードを書いてから、「解答例」を開いて確認しましょう。

---
## 0. 環境準備（JupyterLite 用）

SciPy・NumPy・matplotlib・cvxpy は JupyterLite に同梱されています。日本語グラフ用に `japanize-matplotlib-jlite` を導入します。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["numpy", "scipy", "matplotlib", "japanize-matplotlib-jlite"])
except ImportError:
    pass

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語表示用（plt の後に import する）
from scipy.optimize import minimize_scalar, minimize, linprog, milp, LinearConstraint, Bounds

print(f"SciPy バージョン: {scipy.__version__}")
print(f"NumPy バージョン: {np.__version__}")

---
## 1. 最適化問題とは

最適化問題は、次の 3 つの要素で表されます。

| 要素 | 意味 | 例（生産計画） |
|---|---|---|
| **変数**（決定変数） | 自分で決められる量 | 各製品の生産量 $x_1, x_2$ |
| **目的関数** | 最大化・最小化したい量 | 利潤 $3x_1 + 2x_2$ |
| **制約条件** | 満たさなければならない条件 | 労働時間 $x_1 + x_2 \le 4$ など |

`scipy.optimize` の関数はすべて **最小化** を行います。**最大化** したいときは、目的関数に $-1$ を掛けて最小化します
（利潤の最大化 = 「マイナス利潤」の最小化）。

| 関数 | 用途 |
|---|---|
| `minimize_scalar` | 1 変数の関数の最小化 |
| `minimize` | 多変数の関数の最小化（制約・範囲の指定も可能） |
| `linprog` | 線形計画法（目的関数・制約がすべて 1 次式） |
| `milp` | 混合整数線形計画法（変数が整数） |

---
## 2. 1 変数の最適化：minimize_scalar

逆需要関数 $P = 100 - Q$、総費用 $TC = 100 + 10Q + Q^2/2$ の企業の利潤 $\pi(Q) = PQ - TC$ を最大にする $Q$ を求めます。
SymPy なら一階条件を解いて $Q^* = 30$ ですが、ここでは数値的に探索します。

### 2.1 目的関数を Python の関数として定義する

In [ ]:
def profit(q):
    """生産量 q のときの利潤"""
    price = 100 - q
    total_cost = 100 + 10 * q + 0.5 * q**2
    return price * q - total_cost


def neg_profit(q):
    """最大化 → 「マイナス利潤」の最小化に変換"""
    return -profit(q)


print("Q = 20, 30, 40 のときの利潤:", [profit(q) for q in (20, 30, 40)])

### 2.2 minimize_scalar で解く

`bounded` 法は、探索範囲 `bounds=(下限, 上限)` の中で最小値を探します。
結果オブジェクトの `x` が最適な変数の値、`fun` がそのときの目的関数の値です。

In [ ]:
res = minimize_scalar(neg_profit, bounds=(0, 100), method="bounded")
print(res)
print()
print(f"最適生産量 Q* = {res.x:.4f}")
print(f"最大利潤 = {-res.fun:.4f}")     # 符号を戻すのを忘れずに

In [ ]:
# グラフで確認
q_vals = np.linspace(0, 70, 200)
plt.figure(figsize=(7, 4.5))
plt.plot(q_vals, profit(q_vals), label="利潤 π(Q)")
plt.axvline(res.x, color="red", linestyle="--", label=f"最適生産量 Q* = {res.x:.1f}")
plt.xlabel("生産量 Q")
plt.ylabel("利潤")
plt.title("利潤最大化（1 変数）")
plt.legend()
plt.grid(True)
plt.show()

### 2.3 探索範囲を指定しない場合と、計算コストの確認

`bounds` を省くと Brent 法（黄金分割法の改良版）で探索します。結果の `nfev` は目的関数を評価した回数で、
「何回利潤を計算したか」＝計算コストの目安です。反復法である最適化アルゴリズムは、
数式を解く代わりに、目的関数を何度も評価しながら最適解に近づいていきます。

In [ ]:
res_brent = minimize_scalar(neg_profit)                 # 範囲なし（Brent 法）
res_bounded = minimize_scalar(neg_profit, bounds=(0, 100), method="bounded")
print(f"Brent 法  : Q* = {res_brent.x:.4f}, 関数評価回数 = {res_brent.nfev}")
print(f"bounded 法: Q* = {res_bounded.x:.4f}, 関数評価回数 = {res_bounded.nfev}")

### 練習問題 2

1. 総費用関数 $TC(Q) = 200 + 5Q + 0.1Q^3$ の **平均費用** $AC(Q) = TC/Q$ を最小にする生産量を、$Q \in (1, 50)$ の範囲で求めてください。
2. 逆需要関数 $P = 80 - 0.5Q$、総費用 $TC = 400 + 20Q$ の企業について、利潤を最大にする $Q$ と最大利潤を求めてください。
3. 問 2 で最大利潤が正かどうかを確認し、負なら「操業停止すべき」と表示するコードを書いてください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
res1 = minimize_scalar(lambda q: (200 + 5 * q + 0.1 * q**3) / q, bounds=(1, 50), method="bounded")
print(res1.x, res1.fun)

# 2
res2 = minimize_scalar(lambda q: -((80 - 0.5 * q) * q - (400 + 20 * q)), bounds=(0, 160), method="bounded")
print(res2.x, -res2.fun)

# 3
print("操業継続" if -res2.fun > 0 else "操業停止すべき")
```

</details>

---
## 3. 多変数の最適化：minimize

2 種類の製品を生産する企業の利潤 $\pi(q_1, q_2) = 60q_1 + 80q_2 - (q_1^2 + q_1 q_2 + 2q_2^2)$ を最大化します。
`minimize` には、変数を **1 つの配列** として受け取る関数と、探索の **初期値** `x0` を渡します。

In [ ]:
def neg_profit2(q):
    q1, q2 = q
    return -(60 * q1 + 80 * q2 - (q1**2 + q1 * q2 + 2 * q2**2))


res = minimize(neg_profit2, x0=[1, 1])          # 初期値 (1, 1) から探索（既定は BFGS 法）
print(res)
print()
print(f"最適解: q1 = {res.x[0]:.3f}, q2 = {res.x[1]:.3f}, 最大利潤 = {-res.fun:.3f}")

### 3.1 結果オブジェクトの読み方

| 属性 | 意味 |
|---|---|
| `x` | 最適な変数の値（配列） |
| `fun` | 最適解での目的関数の値 |
| `success` | 収束に成功したか（`True` / `False`） |
| `message` | 終了理由のメッセージ |
| `nit` | 反復回数 |

`success` が `False` のときは、初期値や手法を変えて試す必要があります。

### 3.2 手法（method）を変える

`method` で最適化アルゴリズムを選べます。導関数が不要で頑健な `Nelder-Mead`、
滑らかな関数に速い `BFGS`、範囲制約が使える `L-BFGS-B` などがよく使われます。

In [ ]:
for method in ("Nelder-Mead", "BFGS", "L-BFGS-B", "Powell"):
    r = minimize(neg_profit2, x0=[1, 1], method=method)
    print(f"{method:12s}: x = {np.round(r.x, 3)}, 利潤 = {-r.fun:.3f}, 反復 = {r.nit}, 成功 = {r.success}")

### 3.3 初期値の重要性：複数の極値がある場合

目的関数に山が 2 つある（多峰性）と、初期値によって別の極大値に落ち着くことがあります。
複数の初期値から試して、最も良い解を選ぶのが実践的な対処法です。

In [ ]:
def bumpy(x):
    return -(np.exp(-(x - 1)**2) + 1.5 * np.exp(-(x - 4)**2 / 0.5))


for x0 in (0.0, 2.5, 5.0):
    r = minimize(bumpy, x0=[x0], method="Nelder-Mead")
    print(f"初期値 {x0}: 解 x = {r.x[0]:.3f}, 目的関数 = {r.fun:.3f}")

xs = np.linspace(-1, 6, 300)
plt.figure(figsize=(7, 4))
plt.plot(xs, -bumpy(xs))
plt.title("山が 2 つある関数（初期値によって到達する山が変わる）")
plt.xlabel("x")
plt.grid(True)
plt.show()

### 3.4 探索の経路を可視化する

`callback` 引数に関数を渡すと、反復のたびに現在の解が渡されます。経路を記録して等高線図に重ねると、
アルゴリズムが「山を登っていく」様子が見えます。

In [ ]:
path = []
res = minimize(neg_profit2, x0=[1, 1], method="BFGS", callback=lambda xk: path.append(xk.copy()))
path = np.array([[1, 1]] + path)

q1_grid, q2_grid = np.meshgrid(np.linspace(0, 40, 100), np.linspace(0, 30, 100))
profit_grid = -neg_profit2([q1_grid, q2_grid])

plt.figure(figsize=(7, 5))
plt.contour(q1_grid, q2_grid, profit_grid, levels=20, cmap="viridis")
plt.plot(path[:, 0], path[:, 1], "o-", color="red", label="探索の経路")
plt.scatter([res.x[0]], [res.x[1]], color="black", zorder=5, label="最適解")
plt.xlabel("q1")
plt.ylabel("q2")
plt.title("BFGS 法の探索経路（等高線 = 利潤）")
plt.legend()
plt.grid(True)
plt.show()
print("反復回数:", len(path) - 1)

### 練習問題 3

1. 費用関数 $C(x, y) = (x - 3)^2 + (y - 5)^2 + xy$ を最小にする $(x, y)$ を `minimize` で求めてください。
2. 問 1 を `Nelder-Mead` と `BFGS` の両方で解き、反復回数を比べてください。
3. 利潤関数 $\pi(q_1, q_2) = 100q_1 + 120q_2 - 2q_1^2 - 3q_2^2 - q_1 q_2$ を最大にする生産量と最大利潤を求めてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
r1 = minimize(lambda v: (v[0] - 3)**2 + (v[1] - 5)**2 + v[0] * v[1], x0=[0, 0])
print(np.round(r1.x, 3), round(r1.fun, 3))

# 2
for m in ("Nelder-Mead", "BFGS"):
    r = minimize(lambda v: (v[0] - 3)**2 + (v[1] - 5)**2 + v[0] * v[1], x0=[0, 0], method=m)
    print(m, r.nit)

# 3
r3 = minimize(lambda q: -(100 * q[0] + 120 * q[1] - 2 * q[0]**2 - 3 * q[1]**2 - q[0] * q[1]), x0=[1, 1])
print(np.round(r3.x, 3), round(-r3.fun, 3))
```

</details>

---
## 4. 制約付き最適化：SLSQP と予算制約

### 4.1 予算制約付きの効用最大化

効用関数 $U(x, y) = x^{0.5} y^{0.5}$ を、予算制約 $2x + 3y = 120$ のもとで最大化します。
`minimize` の `constraints` に **等式制約**（`"type": "eq"`）を辞書で渡し、`method="SLSQP"` を指定します。
制約は「$= 0$」の形で書くので、`120 - 2x - 3y` を返す関数にします。

In [ ]:
def neg_utility(v):
    x, y = v
    return -(x**0.5 * y**0.5)


budget = {"type": "eq", "fun": lambda v: 120 - 2 * v[0] - 3 * v[1]}   # 2x + 3y = 120
bounds = [(0.01, None), (0.01, None)]                                    # 消費量は正

res = minimize(neg_utility, x0=[10, 10], method="SLSQP", constraints=[budget], bounds=bounds)
print(f"最適消費量: x = {res.x[0]:.3f}, y = {res.x[1]:.3f}")
print(f"最大効用 = {-res.fun:.4f}, 成功 = {res.success}")
print("解析解（所得の半分ずつを支出）: x = 120/(2*2) = 30, y = 120/(2*3) = 20")

### 4.2 不等式制約

不等式制約は `"type": "ineq"` で、関数の値が **0 以上** になる形で書きます。
たとえば「予算を超えない」$2x + 3y \le 120$ は `120 - 2x - 3y >= 0` です。
「消費量 $x$ は 20 以下」のような条件も追加できます。

In [ ]:
constraints = [
    {"type": "ineq", "fun": lambda v: 120 - 2 * v[0] - 3 * v[1]},   # 予算を超えない
    {"type": "ineq", "fun": lambda v: 20 - v[0]},                    # x は 20 以下（配給制限）
]
res = minimize(neg_utility, x0=[10, 10], method="SLSQP", constraints=constraints, bounds=bounds)
print(f"制限付きの最適消費量: x = {res.x[0]:.3f}, y = {res.x[1]:.3f}, 効用 = {-res.fun:.4f}")
print("支出額:", 2 * res.x[0] + 3 * res.x[1])

### 4.3 無差別曲線と予算線を描く

最適点では、無差別曲線が予算線に接していることを確認しましょう。

In [ ]:
res = minimize(neg_utility, x0=[10, 10], method="SLSQP", constraints=[budget], bounds=bounds)
u_star = -res.fun

x_vals = np.linspace(1, 60, 200)
plt.figure(figsize=(7, 5))
plt.plot(x_vals, (120 - 2 * x_vals) / 3, label="予算線 2x + 3y = 120")
for level in (u_star * 0.7, u_star, u_star * 1.3):
    plt.plot(x_vals, level**2 / x_vals, linestyle="--", label=f"無差別曲線 U = {level:.1f}")
plt.scatter([res.x[0]], [res.x[1]], color="red", zorder=5, label="最適点")
plt.xlim(0, 60)
plt.ylim(0, 45)
plt.xlabel("財 x")
plt.ylabel("財 y")
plt.title("予算制約付き効用最大化")
plt.legend()
plt.grid(True)
plt.show()

### 練習問題 4

1. 効用関数 $U = x^{0.3} y^{0.7}$、予算制約 $4x + 2y = 200$ の最適消費量を SLSQP で求め、解析解（$x = 0.3 \times 200/4 = 15$, $y = 0.7 \times 200/2 = 70$）と比べてください。
2. 生産関数 $F = K^{0.5} L^{0.5}$ で、生産量 $F = 50$ を達成するための費用 $C = 4K + 9L$ を最小にする $(K, L)$ を求めてください（等式制約 `F - 50 = 0`）。
3. 問 1 に「$y$ は 50 以下」という不等式制約を加えて解き直してください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
u1 = lambda v: -(v[0]**0.3 * v[1]**0.7)
r1 = minimize(u1, x0=[10, 10], method="SLSQP", constraints=[{"type": "eq", "fun": lambda v: 200 - 4 * v[0] - 2 * v[1]}], bounds=[(0.01, None)] * 2)
print(np.round(r1.x, 3))

# 2
r2 = minimize(lambda v: 4 * v[0] + 9 * v[1], x0=[10, 10], method="SLSQP", constraints=[{"type": "eq", "fun": lambda v: v[0]**0.5 * v[1]**0.5 - 50}], bounds=[(0.01, None)] * 2)
print(np.round(r2.x, 3), round(r2.fun, 2))

# 3
r3 = minimize(u1, x0=[10, 10], method="SLSQP", constraints=[{"type": "eq", "fun": lambda v: 200 - 4 * v[0] - 2 * v[1]}, {"type": "ineq", "fun": lambda v: 50 - v[1]}], bounds=[(0.01, None)] * 2)
print(np.round(r3.x, 3))
```

</details>

---
## 5. 線形計画法：linprog

目的関数も制約もすべて **1 次式** の最適化問題を **線形計画問題（LP）** といいます。
`linprog` は次の標準形で問題を受け取ります。

$$\min_x\ c^\top x \quad \text{s.t.}\quad A_{ub} x \le b_{ub},\quad A_{eq} x = b_{eq},\quad l \le x \le u$$

### 5.1 生産計画問題

ある工場は製品 A と B を生産します。

| | 製品 A（1 単位） | 製品 B（1 単位） | 利用可能量 |
|---|---|---|---|
| 労働時間 | 1 時間 | 1 時間 | 4 時間 |
| 機械時間 | 2 時間 | 1 時間 | 6 時間 |
| 利潤 | 3 万円 | 2 万円 | |

利潤 $3x_A + 2x_B$ を最大にする生産量を求めます。最大化なので `c` には利潤の符号を反転して渡します。

In [ ]:
c = [-3, -2]                       # 最大化 → 係数の符号を反転
A_ub = [[1, 1],                    # 労働時間の制約
        [2, 1]]                    # 機械時間の制約
b_ub = [4, 6]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)], method="highs")
print(res)
print()
print(f"最適生産量: A = {res.x[0]:.2f}, B = {res.x[1]:.2f}")
print(f"最大利潤 = {-res.fun:.2f} 万円")

### 5.2 実行可能領域を図示する

2 変数の LP は、制約を満たす領域（実行可能領域）の **頂点** のどれかが最適解になります。

In [ ]:
x_vals = np.linspace(0, 5, 100)
plt.figure(figsize=(6.5, 5))
plt.fill_between(x_vals, 0, np.minimum(4 - x_vals, 6 - 2 * x_vals).clip(min=0), alpha=0.3, label="実行可能領域")
plt.plot(x_vals, 4 - x_vals, label="労働時間: x_A + x_B = 4")
plt.plot(x_vals, 6 - 2 * x_vals, label="機械時間: 2x_A + x_B = 6")
for level in (6, 10):
    plt.plot(x_vals, (level - 3 * x_vals) / 2, linestyle="--", color="gray", label=f"利潤 = {level}")
plt.scatter([res.x[0]], [res.x[1]], color="red", zorder=5, label="最適解")
plt.xlim(0, 5)
plt.ylim(0, 6.5)
plt.xlabel("製品 A の生産量")
plt.ylabel("製品 B の生産量")
plt.title("生産計画問題の実行可能領域")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()

### 5.3 輸送問題

2 つの工場から 3 つの店舗へ製品を運びます。輸送費用の合計を最小にする輸送量を決めます。

| 費用（万円/単位） | 店舗 1 | 店舗 2 | 店舗 3 | 供給量 |
|---|---|---|---|---|
| 工場 X | 4 | 6 | 9 | 50 |
| 工場 Y | 5 | 3 | 7 | 40 |
| 需要量 | 30 | 25 | 35 | |

変数は 6 個（工場 × 店舗）です。供給制約は「$\le$ 供給量」、需要制約は「$=$ 需要量」として書きます。

In [ ]:
cost = np.array([[4, 6, 9],
                 [5, 3, 7]])
supply = [50, 40]
demand = [30, 25, 35]

# 変数の並び: x[0..2] = 工場 X → 店舗 1,2,3、x[3..5] = 工場 Y → 店舗 1,2,3
A_supply = [[1, 1, 1, 0, 0, 0],
            [0, 0, 0, 1, 1, 1]]
A_demand = [[1, 0, 0, 1, 0, 0],
            [0, 1, 0, 0, 1, 0],
            [0, 0, 1, 0, 0, 1]]

res_t = linprog(cost.flatten(), A_ub=A_supply, b_ub=supply, A_eq=A_demand, b_eq=demand,
                bounds=[(0, None)] * 6, method="highs")
plan = res_t.x.reshape(2, 3)
print("最適な輸送量:")
for i, factory in enumerate(["工場 X", "工場 Y"]):
    print(f"  {factory}: {np.round(plan[i], 1)}")
print(f"最小輸送費用 = {res_t.fun:.1f} 万円")

### 5.4 食事問題（最小費用で栄養条件を満たす）

歴史的に有名な LP の例です。必要な栄養素を満たしつつ、食費を最小にします。
「栄養素 $\ge$ 必要量」という制約は、両辺に $-1$ を掛けて「$\le$」の形に直します。

In [ ]:
foods = ["米", "鶏肉", "野菜", "牛乳"]
price = np.array([50, 120, 40, 60])            # 円 / 100g
# 栄養素（100g あたり）: エネルギー(kcal), たんぱく質(g), ビタミン(mg)
nutrition = np.array([[350, 6, 0],
                      [200, 20, 1],
                      [30, 2, 40],
                      [65, 3, 5]])
requirement = np.array([2000, 60, 80])         # 1 日の必要量

# 制約: nutrition.T @ x >= requirement  →  -nutrition.T @ x <= -requirement
res_d = linprog(price, A_ub=-nutrition.T, b_ub=-requirement, bounds=[(0, None)] * 4, method="highs")
for food, amount in zip(foods, res_d.x):
    print(f"{food}: {amount * 100:.0f} g")
print(f"最小費用 = {res_d.fun:.0f} 円")
print("摂取栄養素:", np.round(nutrition.T @ res_d.x, 1))

### 5.5 いろいろな制約を linprog の形に直す

| 元の制約 | linprog での書き方 |
|---|---|
| $a x \le b$ | `A_ub`, `b_ub` にそのまま |
| $a x \ge b$ | 両辺に $-1$ を掛けて $-a x \le -b$ |
| $a x = b$ | `A_eq`, `b_eq` |
| $x \ge 0$ | `bounds=(0, None)`（既定値） |
| $x$ が負でもよい | `bounds=(None, None)` |
| 最大化 | `c` の符号を反転し、`-res.fun` を読む |

例: 製品 A を **少なくとも 1 単位** 作り（$x_A \ge 1$）、A と B の **合計をちょうど 3.5 単位** にする（$x_A + x_B = 3.5$）条件を追加します。

In [ ]:
res_mix = linprog([-3, -2],
                  A_ub=[[1, 1], [2, 1], [-1, 0]], b_ub=[4, 6, -1],     # 3 本目は x_A >= 1 を -x_A <= -1 に変換
                  A_eq=[[1, 1]], b_eq=[3.5],
                  bounds=[(0, None), (0, None)], method="highs")
print("最適生産量:", np.round(res_mix.x, 2), "利潤:", -res_mix.fun, "成功:", res_mix.success)

### 練習問題 5

1. 5.1 の生産計画で、製品 A の利潤が 3 万円から 5 万円に上がったときの最適生産量と最大利潤を求めてください。
2. 5.1 に「製品 B は 1 単位以上作る」という制約（`bounds` の下限）を加えて解いてください。
3. 5.3 の輸送問題で、工場 X の供給量が 50 から 30 に減ったときの最小輸送費用を求めてください（需要合計 90 に対して供給合計 70 なので、需要制約を「$\le$」に変え、輸送量の合計 $= 70$ という等式制約を追加してください）。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
r1 = linprog([-5, -2], A_ub=A_ub, b_ub=b_ub, bounds=[(0, None)] * 2, method="highs")
print(np.round(r1.x, 2), -r1.fun)

# 2
r2 = linprog([-3, -2], A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (1, None)], method="highs")
print(np.round(r2.x, 2), -r2.fun)

# 3
r3 = linprog(cost.flatten(), A_ub=A_supply + A_demand, b_ub=[30, 40] + demand,
             A_eq=[[1] * 6], b_eq=[70], bounds=[(0, None)] * 6, method="highs")
print(np.round(r3.x.reshape(2, 3), 1), r3.fun)
```

</details>

---
## 6. 感度分析：シャドウプライス（双対値）

制約の右辺（利用可能な資源量）が 1 単位増えたとき、目的関数がどれだけ改善するかを **シャドウプライス（双対値）** といいます。
「労働時間をあと 1 時間確保できたら、利潤はいくら増えるか」＝「その資源に支払ってもよい上限価格」を表します。

`linprog(method="highs")` の結果では `res.ineqlin.marginals` に不等式制約の双対値が入っています
（最小化問題の値なので、最大化問題では符号を反転して解釈します）。

In [ ]:
res = linprog([-3, -2], A_ub=A_ub, b_ub=b_ub, bounds=[(0, None), (0, None)], method="highs")
shadow = -res.ineqlin.marginals            # 最大化問題なので符号を反転
for name, value, slack in zip(["労働時間", "機械時間"], shadow, res.ineqlin.residual):
    print(f"{name}: シャドウプライス = {value:.2f} 万円/時間, 余り（スラック） = {slack:.2f}")

In [ ]:
# 確認: 労働時間を 4 → 5 に増やして解き直すと、利潤はシャドウプライスの分だけ増える
res_plus = linprog([-3, -2], A_ub=A_ub, b_ub=[5, 6], bounds=[(0, None), (0, None)], method="highs")
print(f"労働時間 4 時間: 利潤 {-res.fun:.2f} 万円")
print(f"労働時間 5 時間: 利潤 {-res_plus.fun:.2f} 万円  → 増分 {-res_plus.fun + res.fun:.2f}")

余り（スラック）が正の制約は「資源が余っている」ので、シャドウプライスは 0 になります
（その資源を増やしても利潤は増えない）。

### 6.1 パラメータを変えて繰り返し解く

利潤係数や資源量を少しずつ変えて解き直すと、最適解がどこで切り替わるかがわかります。

In [ ]:
labor_hours = np.arange(2, 9)
profits = []
for hours in labor_hours:
    r = linprog([-3, -2], A_ub=A_ub, b_ub=[hours, 6], bounds=[(0, None), (0, None)], method="highs")
    profits.append(-r.fun)

plt.figure(figsize=(6.5, 4))
plt.plot(labor_hours, profits, marker="o")
plt.xlabel("利用可能な労働時間")
plt.ylabel("最大利潤（万円）")
plt.title("労働時間と最大利潤の関係（傾き = シャドウプライス）")
plt.grid(True)
plt.show()
print(dict(zip(labor_hours.tolist(), profits)))

### 練習問題 6

1. 5.4 の食事問題について、各栄養素のシャドウプライス（必要量を 1 単位増やしたときの費用増）を表示してください（最小化問題なので符号反転は不要。`res_d.ineqlin.marginals` の符号に注意して解釈してください）。
2. 5.1 の生産計画で、機械時間を 6 から 10 まで 1 ずつ増やしたときの最大利潤を表にしてください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
print(res_d.ineqlin.marginals)   # 制約を -栄養 <= -必要量 と書いたので、値は「必要量 1 増あたりの費用増」の符号反転

# 2
for m in range(6, 11):
    r = linprog([-3, -2], A_ub=A_ub, b_ub=[4, m], bounds=[(0, None)] * 2, method="highs")
    print(m, round(-r.fun, 2))
```

</details>

---
## 7. 整数計画法：milp（プロジェクト選択）

変数が「作る／作らない」（0 か 1）や「台数」（整数）のとき、LP の解を四捨五入しても最適とは限りません。
`milp` は **整数制約** を正しく扱います。

### 7.1 投資プロジェクトの選択（ナップサック問題）

予算 100 億円の範囲で、正味現在価値（NPV）の合計が最大になるようにプロジェクトを選びます。

In [ ]:
projects = ["A", "B", "C", "D", "E"]
npv = np.array([30, 45, 25, 60, 20])       # 各プロジェクトの NPV（億円）
cost = np.array([40, 55, 30, 70, 20])      # 各プロジェクトの必要投資額（億円）
budget = 100

res_m = milp(c=-npv,                                             # NPV 合計の最大化
             constraints=LinearConstraint(cost[np.newaxis, :], ub=budget),   # 予算制約
             integrality=np.ones(len(projects)),                 # すべて整数
             bounds=Bounds(0, 1))                                # 0 か 1
chosen = [p for p, v in zip(projects, res_m.x) if v > 0.5]
print("選ぶプロジェクト:", chosen)
print(f"NPV 合計 = {-res_m.fun:.0f} 億円, 投資額 = {cost @ res_m.x:.0f} 億円")

### 7.2 LP の解を丸めると何が起きるか

同じ問題を整数制約なしで解くと、「プロジェクト D を 0.57 だけ実施する」のような非現実的な解になります。

In [ ]:
res_lp = linprog(-npv, A_ub=cost[np.newaxis, :], b_ub=[budget], bounds=[(0, 1)] * 5, method="highs")
print("LP の解（連続）:", np.round(res_lp.x, 2), "NPV =", round(-res_lp.fun, 1))
print("整数計画の解 :", res_m.x, "NPV =", -res_m.fun)

### 7.3 生産量が整数の生産計画

5.1 の生産計画で生産量を整数に限定してみます。`integrality=[1, 1]` で両方を整数にします。

In [ ]:
res_int = milp(c=[-3, -2], constraints=LinearConstraint(A_ub, ub=b_ub), integrality=[1, 1], bounds=Bounds(0, np.inf))
print("整数の最適生産量:", res_int.x, "利潤 =", -res_int.fun)

### 練習問題 7

1. 7.1 で予算が 120 億円に増えたとき、選ばれるプロジェクトと NPV 合計を求めてください。
2. 7.1 に「プロジェクト A と B は同時には実施できない」（$x_A + x_B \le 1$）という制約を追加して解いてください。
3. 5.3 の輸送問題を、輸送量が整数という条件付きで `milp` で解いてください（供給制約 `LinearConstraint(A_supply, ub=supply)` と需要制約 `LinearConstraint(A_demand, lb=demand, ub=demand)` の 2 つを `constraints` にリストで渡します）。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
r1 = milp(c=-npv, constraints=LinearConstraint(cost[np.newaxis, :], ub=120), integrality=np.ones(5), bounds=Bounds(0, 1))
print([p for p, v in zip(projects, r1.x) if v > 0.5], -r1.fun)

# 2
cons = [LinearConstraint(cost[np.newaxis, :], ub=budget), LinearConstraint([[1, 1, 0, 0, 0]], ub=1)]
r2 = milp(c=-npv, constraints=cons, integrality=np.ones(5), bounds=Bounds(0, 1))
print([p for p, v in zip(projects, r2.x) if v > 0.5], -r2.fun)

# 3
cost_t = np.array([[4, 6, 9], [5, 3, 7]])
cons3 = [LinearConstraint(A_supply, ub=supply), LinearConstraint(A_demand, lb=demand, ub=demand)]
r3 = milp(c=cost_t.flatten(), constraints=cons3, integrality=np.ones(6), bounds=Bounds(0, np.inf))
print(r3.x.reshape(2, 3), r3.fun)
```

</details>

---
## 8. cvxpy でポートフォリオ最適化

**cvxpy** は、最適化問題を数式に近い形で記述できるライブラリです（凸最適化）。
ここでは、5 つの資産に投資するとき、目標収益率を確保しつつ **リスク（分散）を最小化** する配分を求めます
（マーコビッツの平均・分散モデル）。

$$\min_w\ w^\top \Sigma w \quad \text{s.t.}\quad \sum_i w_i = 1,\ w \ge 0,\ \mu^\top w \ge r_{\text{target}}$$

JupyterLite では、ソルバーに `CLARABEL` を指定します。

In [ ]:
import cvxpy as cp

# 5 資産の月次収益率データを乱数で作成（実際には株価データから計算する）
rng = np.random.default_rng(42)
n_assets = 5
true_mean = np.array([0.004, 0.006, 0.008, 0.010, 0.012])
true_vol = np.array([0.02, 0.03, 0.04, 0.05, 0.07])
returns = rng.normal(true_mean, true_vol, size=(120, n_assets))     # 120 か月分

mu = returns.mean(axis=0)              # 期待収益率
Sigma = np.cov(returns, rowvar=False)  # 分散共分散行列
print("期待収益率（月次）:", np.round(mu, 4))
print("標準偏差（月次）  :", np.round(np.sqrt(np.diag(Sigma)), 4))

In [ ]:
w = cp.Variable(n_assets)
target_return = 0.008

problem = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)),
                     [cp.sum(w) == 1, w >= 0, mu @ w >= target_return])
problem.solve(solver=cp.CLARABEL)

print("最適な配分（ウェイト）:", np.round(w.value, 3))
print(f"ポートフォリオの期待収益率 = {mu @ w.value:.4f}, 標準偏差 = {np.sqrt(problem.value):.4f}")

### 8.1 効率的フロンティア

目標収益率を変えながら解き直すと、「その収益率で達成できる最小リスク」の曲線（効率的フロンティア）が描けます。

In [ ]:
targets = np.linspace(mu.min(), mu.max(), 12)
risks = []
for tr in targets:
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), [cp.sum(w) == 1, w >= 0, mu @ w >= tr])
    prob.solve(solver=cp.CLARABEL)
    risks.append(np.sqrt(prob.value))

plt.figure(figsize=(7, 4.5))
plt.plot(risks, targets, marker="o", label="効率的フロンティア")
plt.scatter(np.sqrt(np.diag(Sigma)), mu, color="red", label="個別資産")
plt.xlabel("リスク（標準偏差）")
plt.ylabel("期待収益率")
plt.title("平均・分散ポートフォリオの効率的フロンティア")
plt.legend()
plt.grid(True)
plt.show()

### 8.2 制約を変えてみる：空売りを認める場合

`w >= 0` の制約を外すと、負の配分（空売り）が許されます。同じ目標収益率でもリスクが下がる（または同じ）ことを確認しましょう。
制約が少ないほど最適値は良くなる、というのは最適化問題の一般的な性質です。

In [ ]:
for label, cons in [("空売りなし", [cp.sum(w) == 1, w >= 0, mu @ w >= target_return]),
                    ("空売りあり", [cp.sum(w) == 1, mu @ w >= target_return])]:
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), cons)
    prob.solve(solver=cp.CLARABEL)
    print(f"{label}: 配分 = {np.round(w.value, 3)}, リスク = {np.sqrt(prob.value):.4f}")

### 練習問題 8

1. 目標収益率を 0.010 に上げたときの最適配分とリスクを求めてください。
2. 「1 つの資産への投資比率は 40% 以下」という制約（`w <= 0.4`）を追加して、目標収益率 0.008 で解き直してください。
3. cvxpy で 5.1 の生産計画（線形計画）を解き、`linprog` の結果と一致することを確かめてください（`x = cp.Variable(2)`、`cp.Maximize(3*x[0] + 2*x[1])`）。

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```python
# 1
p1 = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), [cp.sum(w) == 1, w >= 0, mu @ w >= 0.010])
p1.solve(solver=cp.CLARABEL); print(np.round(w.value, 3), np.sqrt(p1.value))

# 2
p2 = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), [cp.sum(w) == 1, w >= 0, w <= 0.4, mu @ w >= 0.008])
p2.solve(solver=cp.CLARABEL); print(np.round(w.value, 3), np.sqrt(p2.value))

# 3
xv = cp.Variable(2)
p3 = cp.Problem(cp.Maximize(3 * xv[0] + 2 * xv[1]), [xv[0] + xv[1] <= 4, 2 * xv[0] + xv[1] <= 6, xv >= 0])
p3.solve(solver=cp.CLARABEL); print(np.round(xv.value, 3), p3.value)
```

</details>

---
## 9. まとめ

| 問題のタイプ | 関数 | ポイント |
|---|---|---|
| 1 変数の最小化・最大化 | `minimize_scalar(f, bounds=, method="bounded")` | 最大化は $-f$ を最小化 |
| 多変数の最適化 | `minimize(f, x0=, method=)` | 初期値 `x0` が必要。`Nelder-Mead`, `BFGS` など |
| 制約付き最適化 | `minimize(..., method="SLSQP", constraints=, bounds=)` | 等式 `"eq"`、不等式 `"ineq"`（$\ge 0$ の形） |
| 線形計画 | `linprog(c, A_ub=, b_ub=, A_eq=, b_eq=, bounds=, method="highs")` | すべて 1 次式。$\le$ の形に直す |
| 感度分析 | `res.ineqlin.marginals`, `res.ineqlin.residual` | シャドウプライスとスラック |
| 整数計画 | `milp(c, constraints=LinearConstraint(...), integrality=, bounds=Bounds(...))` | 0-1 変数・整数変数 |
| 凸最適化 | `cvxpy`（`cp.Variable`, `cp.Problem`, `solve(solver=cp.CLARABEL)`） | ポートフォリオなど 2 次の目的関数 |

## 次のステップ

- `python/sympy/sympy_beginner_tutorial.ipynb` — 記号計算で最適化の一階条件を解析的に解く
- `python/sklearn/sklearn_beginner_tutorial.ipynb` — 機械学習も「損失関数の最小化」という最適化問題
- `python/statsmodels/statsmodels_tutorial.ipynb` — 最小二乗法（残差平方和の最小化）による回帰分析

---
## 総合演習：工場の生産計画と感度分析

ある工場は 3 種類の製品（P1, P2, P3）を生産しています。

| | P1 | P2 | P3 | 利用可能量 |
|---|---|---|---|---|
| 原材料（kg/単位） | 2 | 3 | 4 | 240 |
| 労働時間（時間/単位） | 3 | 2 | 1 | 180 |
| 機械時間（時間/単位） | 1 | 2 | 3 | 150 |
| 利潤（万円/単位） | 5 | 6 | 7 | |

1. 利潤を最大にする各製品の生産量と最大利潤を `linprog` で求めてください。
2. 各資源のシャドウプライスとスラックを表示し、「どの資源を増やすと最も利潤が増えるか」を答えてください。
3. 生産量が整数でなければならないとして `milp` で解き直し、LP の解と比べてください。
4. 労働時間を 180 から 260 まで 20 刻みで増やしたときの最大利潤（LP）をグラフにしてください。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください
profit_coef = np.array([5, 6, 7])
A_res = np.array([[2, 3, 4],
                  [3, 2, 1],
                  [1, 2, 3]])
b_res = np.array([240, 180, 150])

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
profit_coef = np.array([5, 6, 7])
A_res = np.array([[2, 3, 4],
                  [3, 2, 1],
                  [1, 2, 3]])
b_res = np.array([240, 180, 150])
resources = ["原材料", "労働時間", "機械時間"]

# 1. LP
res = linprog(-profit_coef, A_ub=A_res, b_ub=b_res, bounds=[(0, None)] * 3, method="highs")
print("最適生産量 (P1, P2, P3):", np.round(res.x, 2))
print(f"最大利潤 = {-res.fun:.2f} 万円")

# 2. 感度分析
shadow = -res.ineqlin.marginals
for name, sp_, slack in zip(resources, shadow, res.ineqlin.residual):
    print(f"{name}: シャドウプライス = {sp_:.2f}, スラック = {slack:.2f}")
print("最も利潤が増える資源:", resources[int(np.argmax(shadow))])

# 3. 整数計画
res_int = milp(c=-profit_coef, constraints=LinearConstraint(A_res, ub=b_res), integrality=np.ones(3), bounds=Bounds(0, np.inf))
print("整数の最適生産量:", res_int.x, "利潤 =", -res_int.fun)

# 4. 労働時間を変えたときの利潤
hours = np.arange(180, 261, 20)
profits = [-linprog(-profit_coef, A_ub=A_res, b_ub=[240, h, 150], bounds=[(0, None)] * 3, method="highs").fun for h in hours]
plt.figure(figsize=(6.5, 4))
plt.plot(hours, profits, marker="o")
plt.xlabel("労働時間")
plt.ylabel("最大利潤（万円）")
plt.title("労働時間と最大利潤")
plt.grid(True)
plt.show()
print(dict(zip(hours.tolist(), np.round(profits, 2).tolist())))

お疲れさまでした！ 最適化は「変数・目的関数・制約」の 3 つに整理できれば、あとは適切な関数に渡すだけです。
現実の問題（生産計画、配送、投資配分、価格設定）を、この形に落とし込む練習を続けてみてください。